# 08 - RAG Full Service Output

هذا Notebook خاص بإظهار RAG كميزة إضافية:
- FAISS Retrieval
- قراءة الوثائق من SQLite
- بناء Prompt
- Gemini LLM
- Answer + Sources

In [ ]:
from pathlib import Path
import sys, os, json, sqlite3, subprocess, time
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# يستطيع صديقك تغيير هذا المتغير إذا كانت artifacts خارج المشروع.
# مثال في PowerShell قبل فتح notebook:
# $env:IR_ARTIFACT_ROOT="E:\\ir_project_artifacts"
ARTIFACT_ROOT = Path(os.environ.get("IR_ARTIFACT_ROOT", r"E:\ir_project_artifacts"))


def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return Path(paths[0])

DB_PATH = first_existing([
    PROJECT_ROOT / "data" / "documents.sqlite",
    PROJECT_ROOT / "data" / "documents.db",
    ARTIFACT_ROOT / "documents.sqlite",
    ARTIFACT_ROOT / "documents.db",
])

TERRIER_INDEX_PATH = first_existing([
    PROJECT_ROOT / "indexes" / "terrier_medline",
    PROJECT_ROOT / "data" / "indexes" / "terrier_medline",
    ARTIFACT_ROOT / "indexes" / "terrier_medline",
])

BERT_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_bert",
    PROJECT_ROOT / "indexes" / "faiss_bert_full",
    PROJECT_ROOT / "data" / "rag_artifacts",
    ARTIFACT_ROOT / "indexes" / "faiss_bert_full",
    ARTIFACT_ROOT / "indexes" / "faiss_bert",
])

WORD2VEC_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_word2vec",
    PROJECT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec",
])

REPORTS_DIR = PROJECT_ROOT / "reports" / "evaluation_notebook"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT       =", PROJECT_ROOT)
print("ARTIFACT_ROOT      =", ARTIFACT_ROOT)
print("DB_PATH            =", DB_PATH, "exists=", DB_PATH.exists())
print("TERRIER_INDEX_PATH =", TERRIER_INDEX_PATH, "exists=", TERRIER_INDEX_PATH.exists())
print("BERT_INDEX_DIR     =", BERT_INDEX_DIR, "exists=", BERT_INDEX_DIR.exists())
print("WORD2VEC_INDEX_DIR =", WORD2VEC_INDEX_DIR, "exists=", WORD2VEC_INDEX_DIR.exists())

In [ ]:
# فحص ملفات RAG/FAISS أولاً
rag_checks = pd.DataFrame([
    {"name": "DB", "path": str(DB_PATH), "exists": DB_PATH.exists()},
    {"name": "FAISS index", "path": str(BERT_INDEX_DIR / "index.faiss"), "exists": (BERT_INDEX_DIR / "index.faiss").exists()},
    {"name": "doc_ids.pkl", "path": str(BERT_INDEX_DIR / "doc_ids.pkl"), "exists": (BERT_INDEX_DIR / "doc_ids.pkl").exists()},
    {"name": "doc_ids.jsonl", "path": str(BERT_INDEX_DIR / "doc_ids.jsonl"), "exists": (BERT_INDEX_DIR / "doc_ids.jsonl").exists()},
])
display(rag_checks)

In [ ]:
from src.rag.faiss_retriever import BertFaissRetriever

question = "What genes are associated with breast cancer?"

if (BERT_INDEX_DIR / "index.faiss").exists():
    retriever = BertFaissRetriever()
    retrieved = retriever.search(question, top_k=10)
    print("Retrieved documents:", len(retrieved))
    display(pd.DataFrame([{
        "rank": i+1,
        "doc_id": x["doc_id"],
        "score": x["score"],
        "title": x.get("title"),
        "text_sample": (x.get("raw_text") or x.get("abstract") or "")[:250],
    } for i, x in enumerate(retrieved)]))
else:
    print("BERT/RAG FAISS index missing.")

In [ ]:
from src.rag.rag_pipeline import RagPipeline

if (BERT_INDEX_DIR / "index.faiss").exists():
    rag = RagPipeline()
    output = rag.ask(question=question, retrieve_k=10, context_docs=5)
    print("QUESTION:")
    print(output["question"])
    print("\nANSWER:")
    print(output["answer"])
    print("\nSOURCES:")
    for i, source in enumerate(output["sources"], start=1):
        print(f"{i}. doc_id={source['doc_id']} | score={source['score']:.4f} | title={source.get('title')}")
else:
    print("Cannot run RAG because FAISS index is missing.")

In [ ]:
# إظهار الـ Prompt مفيد جداً في التقرير والمقابلة.
if "output" in globals():
    print(output["prompt"][:4000])